In [1]:
import os
import numpy as np
import xarray as xr
import pandas as pd

from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

# ============================
# User settings
# ============================

# Generator site list
gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")

# BARRA-C2 variable paths
u_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/ua100m/latest/"
v_path = "/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/va100m/latest/"


In [2]:
client = Client(n_workers=24,
    threads_per_worker=1,
    memory_limit=f"{int(5)}GB"
)

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 24
Total threads: 24,Total memory: 111.76 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38413,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:32899,Total threads: 1
Dashboard: /proxy/45115/status,Memory: 4.66 GiB
Nanny: tcp://127.0.0.1:32779,


2025-09-15 11:22:13,854 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 7e75eb9af9b19f64f6bf39a60d266988 initialized by task ('rechunk-merge-rechunk-split-where-rechunk-transfer-870d7403a34a7963c185a061ef11d039', 0, 0, 2, 64, 0, 4) executed on worker tcp://127.0.0.1:37143
2025-09-15 11:22:17,012 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 56bbe97c3187bcd8adf4f1eb4d480a64 initialized by task ('rechunk-merge-rechunk-split-where-rechunk-transfer-35c28118154574dff8be6fb251f80e9b', 0, 0, 3, 18, 0, 7) executed on worker tcp://127.0.0.1:37143
2025-09-15 11:22:21,524 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 7e75eb9af9b19f64f6bf39a60d266988 deactivated due to stimulus 'task-finished-1757899341.5039258'
2025-09-15 11:22:21,605 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 56bbe97c3187bcd8adf4f1eb4d480a64 deactivated due to stimulus 'task-finished-1757899341.602814'
2025-09-15 11:22:24,821 - distributed.shuffle._scheduler_plugin - WARNI

Thesse define whether which sample we're calcualting and which model we're using.

In [3]:
sample = 'heatwave'
reanalysis = 'BARRA-C2'

In [4]:
if sample == 'heatwave':
    cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv")
    mode_str = 'heatwave'
    
elif sample == 'baseline':
    cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/no_hw_alpine_cluster.csv")
    mode_str = 'baseline'

if reanalysis == 'BARRA-C2':
    extent = [147.5, 151, -38.5, -33.5]
elif reanalysis == 'BARRA-R2':
    extent = None

lon_min, lon_max, lat_min, lat_max = extent

In [5]:
# Statistically significant w ssmin=20, and Mann-Whitney 
cluster = ['TARALGA1','CROOKWF2',
            'GULLRWF1',
            'GUNNING1',
            'BANGOWF1',
            'WOODLWN1',
            'BOCORWF1']

cluster = gen_csv[gen_csv['DUID'].isin(cluster)][['DUID','lat','lon']]

In [6]:
def get_days(days, nc_dir):
    date_list = pd.to_datetime(days['date'])

    # Build filename filter
    all_files = os.listdir(nc_dir)
    selected_files = [
        os.path.join(nc_dir, f)
        for f in all_files
        if any(d.strftime("%Y%m") in f for d in date_list)
    ]

    # Open multiple files lazily with parallel reads
    ds = xr.open_mfdataset(
        selected_files,
        combine='by_coords',
        parallel=True,
        chunks='auto'
    )

    # Select all hours of the requested dates
    subset = ds.where(ds.time.dt.floor('D').isin(date_list), drop=True)

    return subset

In [7]:
def get_ds():
    u_hw_cluster = get_days(cluster_dates, u_path)
    v_hw_cluster = get_days(cluster_dates, v_path)
    
    ds = xr.merge([u_hw_cluster, v_hw_cluster])
    ds = ds.chunk({'time': -1,'lat': 80,'lon':50})
    
    # Convert to Australia/Sydney
    local_time = (
        pd.DatetimeIndex(ds.time.values)
        .tz_localize("UTC")
        .tz_convert("Australia/Sydney")
    )
    
    # Drop tzinfo so xarray can store it
    local_time_naive = local_time.tz_localize(None)
    
    # Assign back to dataset
    ds = ds.assign_coords(time=local_time_naive)
    return ds


In [8]:
ds = get_ds()
# Crop to extent
ds_subset = ds.sel(**{
    'lon': slice(lon_min, lon_max),
    'lat': slice(lat_min, lat_max)
})
ds_subset = ds_subset.chunk(chunks='auto')

In [9]:
# This computes the composite of variance
ds_subset['windspeed'] = np.sqrt(ds_subset['ua100m']**2 + ds_subset['va100m']**2)

def temporal_variance(x):
    # x has dimensions (time, lat, lon)
    return x.var(dim="time", ddof=1)  # use ddof=1 for unbiased sample variance

hourly_var_per_point = ds_subset['windspeed'].groupby("time.hour").map(temporal_variance)

In [10]:
# This computes the hourly u v field composite.

hourly_composite =  ds_subset.groupby("time.hour").mean()

In [11]:
# This computes the composites of minimum, maximum, and diunal amplitude
def compute_windspeed_composites(windspeed):
    """
    Compute windspeed composites (max, min, amplitude) from u and v.
    
    Parameters
    ----------
    u, v : xarray.DataArray
        Wind vector components (time x lat x lon)
    
    Returns
    -------
    dict
        {"max": DataArray, "min": DataArray, "diff": DataArray}
    """
    # Compute windspeed
    
    # Compute composites along time dimension
    composite_max = windspeed.max(dim="time").compute()
    composite_min = windspeed.min(dim="time").compute()
    diurnal_amp = (composite_max - composite_min).compute()
    
    return composite_max, composite_min, diurnal_amp

max_speed, min_speed, diurnal_amp = compute_windspeed_composites(ds_subset['windspeed'])

In [12]:
# --- Assemble into one Dataset ---
results = xr.Dataset(
    {
        "windspeed_variance": hourly_var_per_point,
        "windspeed_mean": hourly_composite["windspeed"],
        "u_mean": hourly_composite["ua100m"],
        "v_mean": hourly_composite["va100m"],
        "windspeed_max": max_speed,
        "windspeed_min": min_speed,
        "diurnal_amp": diurnal_amp,
    }
)

results = results.assign_attrs(reanalysis=reanalysis, mode=mode_str)

# Write to NetCDF lazily with dask
with ProgressBar():
    results.to_netcdf(f"/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_{reanalysis}_{mode_str}_composites.nc", compute=True)

In [13]:
xr.open_dataset(f"/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_{reanalysis}_{mode_str}_composites.nc")

<xarray.Dataset> Size: 9MB
Dimensions:             (lon: 88, lat: 125, hour: 24)
Coordinates:
    height              float64 8B ...
  * lon                 (lon) float64 704B 147.5 147.5 147.6 ... 150.9 151.0
  * lat                 (lat) float64 1kB -38.49 -38.45 -38.41 ... -33.57 -33.53
    crs                 int32 4B ...
  * hour                (hour) int64 192B 0 1 2 3 4 5 6 ... 17 18 19 20 21 22 23
Data variables:
    windspeed_variance  (hour, lat, lon) float64 2MB ...
    windspeed_mean      (hour, lat, lon) float64 2MB ...
    u_mean              (hour, lat, lon) float64 2MB ...
    v_mean              (hour, lat, lon) float64 2MB ...
    windspeed_max       (lat, lon) float64 88kB ...
    windspeed_min       (lat, lon) float64 88kB ...
    diurnal_amp         (lat, lon) float64 88kB ...
Attributes:
    reanalysis:  BARRA-C2
    mode:        heatwave